<a href="https://colab.research.google.com/github/aa-ahmed-arif/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aa-ahmed-arif/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

I will prioritize content pages that are potentially worth refreshing based on two observable signals: staleness and search volume.

Pages that have not been updated for a longer time receive more priority, while pages with higher search impressions receive more priority because a refresh may affect a larger amount of search traffic.

The rule is a baseline, not a claim that these signals cause decline. It is intended to create a transparent review queue that can later be compared against an ML model.

### Reason codes

- STALE_HIGH_VOLUME — page is relatively old and has relatively high search impressions.
- STALE — page is relatively old but has lower search volume.
- HIGH_VOLUME — page has high search volume but is not especially stale.
- REVIEW — does not strongly match either condition but remains in the queue for completeness.

### Signal verdicts

Staleness (days_since_last_update) — CONFIRMED: the data shows a clear difference between recently updated and stale pages, so this is a reasonable signal for a refresh-priority baseline.
Search volume (impressions_90d) — CONFIRMED: pages vary substantially in search impressions, so volume is a reasonable signal for deciding which refresh opportunities may have greater potential impact.

In [5]:
import os
import pandas as pd
import numpy as np

# Make sure we are in the repository
REPO_DIR = "/content/flyrank-ml-internship"

if not os.path.exists(REPO_DIR):
    import subprocess
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/aa-ahmed-arif/flyrank-ml-internship.git",
            REPO_DIR
        ],
        check=True
    )

os.chdir(REPO_DIR)

# Load starter dataset
DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))

# Two signals used by the rule
stale_threshold = df["days_since_last_update"].median()
volume_threshold = df["impressions_90d"].median()

# Signal 1: staleness
df["stale_bucket"] = np.where(
    df["days_since_last_update"] >= stale_threshold,
    "Stale",
    "Recently updated"
)

print("\nStaleness signal check:")
print(
    df["stale_bucket"]
    .value_counts()
    .rename_axis("bucket")
    .reset_index(name="n")
    .to_string(index=False)
)

# Signal 2: search volume
df["volume_bucket"] = np.where(
    df["impressions_90d"] >= volume_threshold,
    "High volume",
    "Lower volume"
)

print("\nSearch-volume signal check:")
print(
    df["volume_bucket"]
    .value_counts()
    .rename_axis("bucket")
    .reset_index(name="n")
    .to_string(index=False)
)

print("\nThresholds:")
print("Staleness median:", stale_threshold)
print("Search-volume median:", volume_threshold)

Rows: 30000
Columns: 44

Staleness signal check:
          bucket     n
           Stale 25707
Recently updated  4293

Search-volume signal check:
      bucket     n
 High volume 15007
Lower volume 14993

Thresholds:
Staleness median: 20.0
Search-volume median: 731.0


## 2. Build the ranked queue (writes the CSV)

The baseline score combines the two selected signals.

A page receives one point for being at or above the median staleness and one point for being at or above the median search volume. The resulting score ranges from 0 to 2.

Pages with a score of 2 are given the highest priority because they are both relatively stale and relatively high-volume. Pages with a score of 1 receive lower priority, and pages with a score of 0 are lowest priority.

The rule uses only information available in the starter snapshot and does not use the declining label, trend direction, or any future outcome.

In [6]:
# Build the baseline score
baseline = df.copy()

baseline["stale_flag"] = (
    baseline["days_since_last_update"] >= stale_threshold
).astype(int)

baseline["high_volume_flag"] = (
    baseline["impressions_90d"] >= volume_threshold
).astype(int)

baseline["baseline_score"] = (
    baseline["stale_flag"] +
    baseline["high_volume_flag"]
)

# Assign one reason code
baseline["reason_code"] = np.select(
    [
        (baseline["stale_flag"] == 1) & (baseline["high_volume_flag"] == 1),
        (baseline["stale_flag"] == 1),
        (baseline["high_volume_flag"] == 1)
    ],
    [
        "STALE_HIGH_VOLUME",
        "STALE",
        "HIGH_VOLUME"
    ],
    default="REVIEW"
)

# Action label
baseline["action"] = np.where(
    baseline["baseline_score"] == 2,
    "PRIORITIZE_REFRESH",
    np.where(
        baseline["baseline_score"] == 1,
        "REVIEW",
        "LOW_PRIORITY"
    )
)

# Rank highest score first, then higher impressions, then greater staleness
baseline = baseline.sort_values(
    ["baseline_score", "impressions_90d", "days_since_last_update"],
    ascending=[False, False, False]
).reset_index(drop=True)

baseline["rank"] = baseline.index + 1

# Keep useful fields in the final queue
output_columns = [
    "rank",
    "content_id",
    "content_type",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "baseline_score",
    "reason_code",
    "action"
]

queue = baseline[output_columns].copy()

# Create output directory
os.makedirs("work/outputs", exist_ok=True)

# Write the required CSV
output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("Queue written to:", output_path)
print("Total ranked pages:", len(queue))

print("\nReason-code counts:")
print(queue["reason_code"].value_counts())

print("\nTop 10:")
print(queue.head(10).to_string(index=False))

Queue written to: work/outputs/baseline_action_score.csv
Total ranked pages: 30000

Reason-code counts:
reason_code
STALE_HIGH_VOLUME    13040
STALE                12667
REVIEW                2326
HIGH_VOLUME           1967
Name: count, dtype: int64

Top 10:
 rank           content_id    content_type  content_age_days  days_since_last_update  impressions_90d  avg_position  ctr  baseline_score       reason_code             action
    1 content_5fe46e04994d keyword article               537                     104           517715           4.2 0.14               2 STALE_HIGH_VOLUME PRIORITIZE_REFRESH
    2 content_aaef01a50def keyword article               445                      22           517109           5.4 0.25               2 STALE_HIGH_VOLUME PRIORITIZE_REFRESH
    3 content_8c19996aa890 keyword article               445                      20           509252           2.5 0.15               2 STALE_HIGH_VOLUME PRIORITIZE_REFRESH
    4 content_2cb567c3c89b keyword article   

## 3. Top-20 review

The top 20 pages are reviewed as decision-support rather than treated as automatically correct recommendations.

For each page, the action and reason code come directly from the baseline rule. The confidence note explains why the page was selected, while the "what would make it wrong" note identifies information that the simple rule does not capture.

A high score therefore means "high priority for review", not "definitely needs a refresh".

In [7]:
# Generate a review table for the top 20
top20 = queue.head(20).copy()

def confidence_note(row):
    if row["baseline_score"] == 2:
        return "Both staleness and search volume are high relative to the dataset."
    elif row["baseline_score"] == 1:
        return "Only one of the two baseline signals is high."
    return "Neither baseline signal is high."

def wrongness_note(row):
    return (
        "Could be wrong if the page is intentionally evergreen, "
        "has strong recent performance, or has other business context "
        "not captured by these two signals."
    )

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(wrongness_note, axis=1)

review_columns = [
    "rank",
    "content_id",
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

print(top20[review_columns].to_string(index=False))

 rank           content_id             action       reason_code                                                    confidence_note                                                                                                                               what_would_make_it_wrong
    1 content_5fe46e04994d PRIORITIZE_REFRESH STALE_HIGH_VOLUME Both staleness and search volume are high relative to the dataset. Could be wrong if the page is intentionally evergreen, has strong recent performance, or has other business context not captured by these two signals.
    2 content_aaef01a50def PRIORITIZE_REFRESH STALE_HIGH_VOLUME Both staleness and search volume are high relative to the dataset. Could be wrong if the page is intentionally evergreen, has strong recent performance, or has other business context not captured by these two signals.
    3 content_8c19996aa890 PRIORITIZE_REFRESH STALE_HIGH_VOLUME Both staleness and search volume are high relative to the dataset. Could be wrong if the p

## 4. Weak picks + leakage check

The weaker selections show a limitation of the simple rule. A page can receive priority because it has high search volume even when it was updated recently, or because it is stale even when its search volume is relatively low. These cases may not actually be good refresh opportunities.

The baseline deliberately does not use the declining label, trend_direction, or trend_pct. It also does not use future-window information. The score therefore uses only observable snapshot signals available at the decision moment.

This is a transparent decision-support baseline, not a claim that the selected pages definitely need a refresh. A later ML model can be compared against this baseline to test whether combining more signals improves the ranking.


In [8]:
# Show some weaker/less convincing selections
weak_picks = queue[
    queue["reason_code"].isin(["STALE", "HIGH_VOLUME", "REVIEW"])
].head(10)

print("Examples of weaker baseline selections:")
print(
    weak_picks[
        [
            "rank",
            "content_id",
            "days_since_last_update",
            "impressions_90d",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ].to_string(index=False)
)

# Leakage check
leakage_columns = [
    "is_declining_label",
    "trend_direction",
    "trend_pct"
]

print("\nLeakage check:")
for col in leakage_columns:
    print(
        col,
        "used in baseline:",
        col in queue.columns
    )

print("\nBaseline feature columns:")
print([
    "days_since_last_update",
    "impressions_90d"
])

print("\nFinal queue shape:", queue.shape)
print("CSV exists:", os.path.exists(output_path))

Examples of weaker baseline selections:
 rank           content_id  days_since_last_update  impressions_90d  baseline_score reason_code action
13041 content_07e0b9af8b1a                       8           214816               1 HIGH_VOLUME REVIEW
13042 content_8ba747cf969e                       8           152968               1 HIGH_VOLUME REVIEW
13043 content_e12868d1f396                       7           149712               1 HIGH_VOLUME REVIEW
13044 content_ae2af765d276                       8           129864               1 HIGH_VOLUME REVIEW
13045 content_7760b682ccf3                      15           121422               1 HIGH_VOLUME REVIEW
13046 content_b6100a228edc                      14           121045               1 HIGH_VOLUME REVIEW
13047 content_0919dd345d80                       7           119217               1 HIGH_VOLUME REVIEW
13048 content_33da44cb09c9                      14           114528               1 HIGH_VOLUME REVIEW
13049 content_3f38d9d022e2       

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.